# 第 7 周：QLoRA 微调 – 在特定任务上优于前沿模型

**公关焦点：**本笔记本展示了*对 QLoRA* 的理解和实现：我们在**特定任务**（通过医患对话生成临床记录）上训练**开源模型**，并**将其与前沿模型**（通过 OpenRouter 的 GPT-4）进行比较，以尝试**在该任务上**优于它**。

- **任务：** 根据对话，生成结构化的临床记录（范围狭窄、定义明确的任务）。
- **QLoRA：** 4 位量化基本模型（NF4，双量化）+ LoRA 适配器 — 在微调时保持低内存。
- **LoRA：** 低阶自适应（例如 `q_proj`、`v_proj`）；仅训练了约 0.2% 的参数。
- **优于前沿：** 我们在相同的评估集上评估我们的微调模型与 GPT-4 的比较；在特定于任务的数据上，小型调整模型可以匹配或击败通才前沿模型。

**型号：** 默认 TinyLlama（无需注册）。可选：[LLaMA 3.2](https://huggingface.co/meta-llama/Llama-3.2-3B) — 设置 `HF_TOKEN` 和 `MODEL_NAME`。

**你需要做什么：**
1. **按顺序运行：** 运行安装单元（单元 1）一次，然后使用 **Kernel → Run All** 以便定义每个变量。
2. **型号：** 默认为 TinyLlama（无需注册）。对于 LLaMA 3.2：通过上面的链接获取访问权限，然后设置 env `HF_TOKEN` 并更改配置单元中的 `MODEL_NAME`。
3. **权重和偏差：** 可选。训练无需它即可进行。要登录到 W&B，请在训练参数中设置 `report_to="wandb"` 并在下一个单元格中运行 `wandb.login()`。
4. **OpenRouter（GPT-4 比较）：** 可选。仅当您想与前沿模型进行比较时才设置 env `OPENROUTER_API_KEY`。
5. **GPU：** 训练需要 GPU（例如 Colab T4/A100）。在 CPU 上，它会非常慢或 OOM。

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
# 首先安装 deps（运行此单元一次，然后运行全部）。如果 pef 仍然丢失，请重新启动内核。
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "torch", "transformers", "accelerate", "peft", "trl", "bitsandbytes", "datasets", "wandb", "gradio", "requests"])

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer
from datasets import load_dataset
import wandb
import json
import os
import numpy as np
from tqdm import tqdm
import requests
import gradio as gr

In [ ]:
# --- QLoRA + LoRA 配置（开源模型，具体任务，尝试突破前沿）---
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # or "meta-llama/Llama-3.2-3B" + HF_TOKEN
HF_TOKEN = os.getenv("HF_TOKEN")

# 任务：通过对话生成临床记录（与 GPT-4 进行比较的特定任务）
DATASET_NAME = "ClinicianFOCUS/ACI-Bench-Refined"
OUTPUT_DIR = "./lora-medical-llama"
WANDB_PROJECT = "qlora-medical-extraction"

# LoRA：注意力投射的低阶适配器（小型可训练集）
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.1
TARGET_MODULES = ["q_proj", "v_proj"]

# 训练
LEARNING_RATE = 2e-4
BATCH_SIZE = 4
GRAD_ACC_STEPS = 4
EPOCHS = 3
MAX_SEQ_LENGTH = 512
WARMUP_STEPS = 100
LOGGING_STEPS = 10
EVAL_STEPS = 200
SAVE_STEPS = 200

# QLoRA：4 位量化（NF4 + 双量化），因此基本模型适合低内存
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# OpenRouter（可选 - 仅在稍后进行前沿模型比较时需要）
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
dataset = load_dataset(DATASET_NAME, token=HF_TOKEN)
train_dataset = dataset["train"]

# 创建验证分割 (90/10)
split_dataset = train_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

def format_instruction(example):
    instruction = f"Generate a structured clinical note from the following doctor-patient dialogue:\n{example['dialogue']}"
    response = example['note']
    return f"### Instruction:\n{instruction}\n\n### Response:\n{response}"

train_dataset = train_dataset.map(lambda x: {"text": format_instruction(x)})
eval_dataset = eval_dataset.map(lambda x: {"text": format_instruction(x)})

print(f"Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# 在 MPS (Apple Silicon) 上，不支持 4 位 + 8 位优化器 → 使用 bf16 + LoRA（无 QLoRA）
_use_cuda = torch.cuda.is_available()
_is_mps = getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available()
_use_qlora = _use_cuda  # QLoRA only on CUDA; on MPS use full bf16 + LoRA so training runs

if _use_qlora:
    # QLoRA：4 位基础 + LoRA (CUDA)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        token=HF_TOKEN,
    )
    model = prepare_model_for_kbit_training(model)
else:
    # 仅限 LoRA：bf16 基础（MPS/CPU – 避免位和字节 8 位优化器）
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
        token=HF_TOKEN,
    )

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
if _is_mps:
    print("(MPS: using bf16 + LoRA; run on Colab/CUDA for QLoRA.)")

In [ ]:
# MPS：必须使用 adamw_torch（bitsandbytes 8 位未实现）； CUDA：可以使用paged_adamw_8bit
_optim = "paged_adamw_8bit" if _use_cuda else "adamw_torch"
_dataloader_pin_memory = _use_cuda

# SFTConfig (TRL) 包括训练参数 + SFT 特定的：dataset_text_field、max_length
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACC_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    num_train_epochs=EPOCHS,
    logging_steps=LOGGING_STEPS,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=True,
    optim=_optim,
    dataloader_pin_memory=_dataloader_pin_memory,
    report_to="none",
    run_name="medical-llama-qlora",
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
)

In [ ]:
# 可选：如果您在上面设置了 report_to="wandb"，请运行 wandb.login() 并在出现提示时粘贴您的 API 密钥
# 万db.login()

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
# wandb.finish() # 仅当您使用 report_to="wandb" 时

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
def evaluate_model(model, tokenizer, eval_dataset, num_samples=50):
    model.eval()
    correct = 0
    total = 0
    for example in tqdm(eval_dataset.select(range(min(num_samples, len(eval_dataset))))):
        prompt = example["text"].split("### Response:\n")[0] + "### Response:\n"
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.1)
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        if "### Response:\n" in response:
            pred = response.split("### Response:\n")[-1].strip()
        else:
            pred = ""
        ref = example["text"].split("### Response:\n")[-1].strip()
        # 简单的启发式：检查是否像注释一样（可以改进）
        if "note" in pred.lower() and len(pred) > 50:
            correct += 1
        total += 1
    return correct / total if total > 0 else 0

accuracy = evaluate_model(model, tokenizer, eval_dataset)
print(f"Validation accuracy: {accuracy:.4f}")

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
def query_openrouter(prompt, model="openai/gpt-4"):
    response = requests.post(
        url="https://openrouter.ai/api/v1/chat/completions",
        headers={"Authorization": f"Bearer {OPENROUTER_API_KEY}"},
        json={
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.1,
            "max_tokens": 512,
        }
    )
    return response.json()["choices"][0]["message"]["content"]

subset_size = 20
eval_subset = eval_dataset.select(range(min(subset_size, len(eval_dataset))))
gpt4_accuracy = None
if OPENROUTER_API_KEY:
    gpt4_correct = 0
    for example in tqdm(eval_subset):
        prompt = example["text"].split("### Response:\n")[0] + "### Response:\n"
        try:
            gpt4_pred = query_openrouter(prompt, model="openai/gpt-4")
        except Exception:
            gpt4_pred = ""
        if "note" in gpt4_pred.lower() and len(gpt4_pred) > 50:
            gpt4_correct += 1
    gpt4_accuracy = gpt4_correct / subset_size
    print(f"GPT-4 accuracy: {gpt4_accuracy:.4f}")
print(f"Your QLoRA (fine-tuned open-source) accuracy: {accuracy:.4f}")
if gpt4_accuracy is not None:
    print(f"→ Outperform frontier? {'Yes' if accuracy >= gpt4_accuracy else 'Close / No'} (vs GPT-4 {gpt4_accuracy:.4f})")

In [ ]:
from peft import PeftModel

# 重新加载基础模型 + LoRA 权重进行推理
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
    token=HF_TOKEN,
)
tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
model.eval()

def generate_response(dialogue, temperature=0.1, max_new_tokens=256):
    prompt = f"### Instruction:\nGenerate a structured clinical note from the following doctor-patient dialogue:\n{dialogue}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "### Response:\n" in response:
        response = response.split("### Response:\n")[-1].strip()
    return response

iface = gr.Interface(
    fn=generate_response,
    inputs=[
        gr.Textbox(lines=5, placeholder="Enter doctor-patient dialogue...", label="Dialogue"),
        gr.Slider(0.0, 1.0, value=0.1, label="Temperature"),
        gr.Slider(64, 512, value=256, step=32, label="Max New Tokens")
    ],
    outputs=gr.Textbox(lines=10, label="Generated Clinical Note"),
    title="Medical Note Generator (QLoRA fine-tuned open-source model)",
    description="Enter a dialogue between doctor and patient to generate a structured clinical note."
)

# 无头/无显示：share=True 给出公共 URL； inbrowser=False 跳过打开浏览器
iface.launch(share=True, inbrowser=False)
# 在任何设备上的任何浏览器中打开打印的 URL（例如 https://xxx.gradio.live）。